# Z24 Mamba baseline — native 6,000-point segments

This experiment uses `inputs.npy` directly: 1,530 segments × 27 sensors × 6,000 time points. The recommended hard split holds out complete measurement setups: setups 0–4 train, 5–6 validation, and 7–8 test. Ten segment probabilities are averaged into one prediction per original `(scenario, setup)` recording, so checkpoint selection and primary metrics are recording-level.

In [ ]:
%cd /kaggle/working
!rm -rf time-series-project
!git clone https://github.com/TuanHungGiang/time-series-project.git
%cd time-series-project
!pip install -q huggingface_hub scikit-learn matplotlib

from huggingface_hub import snapshot_download

DATA_DIR = snapshot_download(
    repo_id="thanglexuan/Z24-dataset-processed",
    repo_type="dataset",
    local_dir="Z24-dataset-processed",
    allow_patterns=["inputs.npy", "labels.npy"],
)
print("Dataset saved at:", DATA_DIR)

## Hard baseline

Augmentation is applied on the fly to training batches only. Validation and test recordings are normalized with train statistics but are never augmented. Conservative settings are used to avoid strongly shifting the vibration frequencies.

In [ ]:
!python -u run_prism_z24.py \
  --model mamba \
  --split setup_holdout \
  --epochs 30 \
  --patience 10 \
  --batch_size 16 \
  --lr 3e-4 \
  --lr_schedule cosine \
  --weight_decay 1e-4 \
  --dropout 0.1 \
  --norm robust \
  --stem_stride 25 \
  --aug_noise 0.03 \
  --aug_warp 0.01 \
  --aug_crop_min 0.9 \
  --seed 42

For a harder low-data stress test after establishing the baseline, change only `--split setup_holdout` to `--split low_data`. That protocol uses setups 0–2 / 3–4 / 5–8 for train/validation/test.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

run_dir = Path("results/mamba_setup_holdout_cosine_aug_seed42")
metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2))

for name in ["training_curves.png", "confusion_matrix.png", "roc_curve.png",
             "tsne_2d.png", "tsne_3d.png"]:
    display(Image(filename=str(run_dir / name)))

## Recording-level accuracy by held-out setup

Every row in `predictions_test_recording.npz` is one independent source recording, obtained by averaging its ten segment probabilities.

In [ ]:
import numpy as np
import pandas as pd

pred = np.load(run_dir / "predictions_test_recording.npz")
df = pd.DataFrame({
    "true_class": pred["y_true"],
    "predicted_class": pred["y_pred"],
    "setup": pred["setup"],
    "segments_aggregated": pred["segment_count"],
})
df["correct"] = (df["true_class"] == df["predicted_class"]).astype(int)
display(df)
display((100 * df.pivot(index="true_class", columns="setup", values="correct"))
        .rename_axis(columns="test_setup"))